In [6]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import time

BASE_URL = "https://www.gutenberg.org"

In [ ]:
def get_soup(url):
    response = requests.get(url)

    response.raise_for_status()

    return BeautifulSoup(response.text, "html.parser")


def get_books_from_category(category_url):
    soup = get_soup(category_url)

    books = []

    for link in soup.find_all("a", href=True):

        href = link["href"]

        match = re.fullmatch(r"/ebooks/(\d+)", href)

        if not match:
            continue

        book_id = match.group(1)

        title = link.get_text(" ", strip=True)

        books.append({
            "book_id": book_id,
            "title": title,
            "ebook_url": urljoin(BASE_URL, href)
        })

    return books

def get_metadata_table(soup):
    metadata = {}

    for row in soup.select("table tr"):
        cells = row.find_all(["th", "td"])

        if len(cells) < 2:
            continue

        key = cells[0].get_text(" ", strip=True)
        value = cells[1].get_text(" ", strip=True)

        if key == "Subject":
            metadata.setdefault("Subject", []).append(value)
        else:
            metadata[key] = value

    return metadata

def get_summary(soup):
    page_body = soup.find("div", class_="page-body")

    if not page_body:
        return None

    text = page_body.get_text(" ", strip=True)

    text = re.sub(r"^QR code\s*", "", text)

    summary = text.split("Read more", 1)[0]

    summary = re.sub(
        r"\(This is an automatically generated summary\.\)",
        "",
        summary
    )

    return summary.strip()

def get_book_metadata(book):

    print(f"Coletando: {book['title']}")

    soup = get_soup(book["ebook_url"])

    metadata = get_metadata_table(soup)

    data = {
        "book_id": book["book_id"],
        "title": metadata.get("Title", book["title"]),
        "author": metadata.get("Author"),
        "language": metadata.get("Language"),
        "loc_class": metadata.get("LoC Class"),
        "subjects": metadata.get("Subject", []),
        "release_date": metadata.get("Release Date"),
        "ebook_url": book["ebook_url"],
        "summary": get_summary(soup)
    }

    data["subjects"] = " | ".join(data["subjects"])

    return data



In [ ]:
def main():

    category_url = (
        "https://www.gutenberg.org/ebooks/bookshelf/645"
    )

    books = get_books_from_category(category_url)

    print(f"Livros encontrados: {len(books)}")

    results = []

    for book in books[:5]:

        try:

            data = get_book_metadata(book)

            results.append(data)

            time.sleep(5)

        except requests.RequestException as error:

            print(
                f"Erro ao coletar {book['title']}: {error}"
            )

    df = pd.DataFrame(results)

    df.to_csv(
        "gutenberg_books.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("CSV criado com sucesso.")


if __name__ == "__main__":
    main()

Livros encontrados: 25
Coletando: Pride and Prejudice Jane Austen 177409 downloads
Coletando: Moby Dick; Or, The Whale Herman Melville 161981 downloads
Coletando: Crime and Punishment Fyodor Dostoyevsky 111160 downloads
Coletando: Frankenstein; or, the modern prometheus Mary Wollstonecraft Shelley 106091 downloads
Coletando: Alice's Adventures in Wonderland Lewis Carroll 105270 downloads
CSV criado com sucesso.


In [12]:
import pandas as pd

df = pd.read_csv("gutenberg_books.csv")

df

,book_id,title,author,language,loc_class,subjects,release_date,ebook_url,summary
0,1342,Pride and Prejudice,"Austen, Jane, 1775-1817",English,PR: Language and Literatures: English literature,England -- Fiction | Young women -- Fiction | ...,"Jun 1, 1998",https://www.gutenberg.org/ebooks/1342,"For your e-reader or reading app — Kindle, Kob..."
1,2701,"Moby Dick; Or, The Whale","Melville, Herman, 1819-1891",English,PS: Language and Literatures: American and Can...,Whaling -- Fiction | Sea stories | Psychologic...,"Jul 1, 2001",https://www.gutenberg.org/ebooks/2701,"For your e-reader or reading app — Kindle, Kob..."
2,2554,Crime and Punishment,"Dostoyevsky, Fyodor, 1821-1881",English,PG: Language and Literatures: Slavic (includin...,Detective and mystery stories | Psychological ...,"Mar 28, 2006",https://www.gutenberg.org/ebooks/2554,"For your e-reader or reading app — Kindle, Kob..."
3,84,"Frankenstein; or, the modern prometheus","Shelley, Mary Wollstonecraft, 1797-1851",English,PR: Language and Literatures: English literature,Science fiction | Horror tales | Gothic fictio...,"Oct 1, 1993",https://www.gutenberg.org/ebooks/84,"For your e-reader or reading app — Kindle, Kob..."
4,11,Alice's Adventures in Wonderland,"Carroll, Lewis, 1832-1898",English,PZ: Language and Literatures: Juvenile belles ...,Fantasy fiction | Children's stories | Imagina...,"Jun 27, 2008",https://www.gutenberg.org/ebooks/11,"For your e-reader or reading app — Kindle, Kob..."
